# Mission 1 — WavLM + Wav2Vec2 학습 기록

고정된 음성 특징에서 Scaler와 Logistic Regression을 학습하고 Train 3-fold OOF로 확률 결합 비중을 선택한 실험이다. WavLM C=0.1, Wav2Vec2 C=1.0, 최종 비중은 75:25이다. 음성 모델 본체는 fine-tuning하지 않았다.

이 노트북의 학습·평가 출력은 사용자가 제공한 실행본에서 그대로 보존했다. 누락된 환경 준비 셀을 복원하고 실제 실행 runner를 읽을 수 있는 소스로 포함했다. 복원 셀은 아직 재실행하지 않았으며 실행 횟수나 출력을 만들어 넣지 않았다. 캐시 무결성·재개·평가 누수 방지 코드는 결과 재현에 필요하여 유지했다.

추론은 이 노트북이나 특징 캐시를 사용하지 않는다. 같은 폴더의 `model.pt`, `model.py`, `inference.py`, `requirements.txt`로 실행한다.

```bash
python inference.py --audio_dir /path/to/wav --label_dir /path/to/json --ckpt_path model.pt --output ./outputs/mission1.csv
```

원 실험 정밀도를 유지하는 추론 실행 환경은 CUDA/BF16 지원 GPU이다. Mac에서 체크포인트 복원 및 원 LR 확률 일치는 확인했지만 CUDA/BF16 전체 WAV 추론은 검증하지 않았다.


## 1. Drive 연결 — 원 실행 출력 보존

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True))


Mounted at /content/drive
NVIDIA L4



## 2. 실행 환경 복원

In [ ]:
# 기존 실행과 동일한 CPU 학습 패키지를 격리된 환경에 설치한다.
from pathlib import Path
import sys, subprocess, hashlib, urllib.request
WORK = Path('/content/mission1_training')
WORK.mkdir(parents=True, exist_ok=True)
PY = WORK / 'venv/bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(WORK/'venv')], check=True)
subprocess.run([str(PY), '-m', 'pip', 'install',
    'numpy==2.2.2', 'scipy==1.15.3', 'scikit-learn==1.6.1',
    'joblib==1.4.2', 'threadpoolctl==3.6.0'], check=True)
# 고정된 학습/내부 검증 분할을 가져온 후 내용 해시를 확인한다.
split = WORK / 'split_assignments.csv'
urllib.request.urlretrieve('https://raw.githubusercontent.com/eunkyo00/DCC2026-AIGO/591c0b3/mission-1/validation/split_assignments.csv', split)
assert hashlib.sha256(split.read_bytes()).hexdigest() == '04b5018778321f775a0bf95949a37636090d4df4e3710b16627dfee309408f06'


## 3. 실제 학습 코드

원 실행 runner와 SHA256이 일치한다. Scaler/LR 학습, Train OOF 비중 선택, 설정 고정 후 평가를 포함한다.

In [ ]:
%%writefile /content/mission1_training/run.py
"""Cache-only, Train-OOF-selected WavLM/Wav2Vec2 probability ensemble."""
import argparse
from concurrent.futures import ThreadPoolExecutor
import csv
import hashlib
import io
import json
import os
from pathlib import Path
import platform
import time
import warnings
import zipfile

import joblib
import numpy as np
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

SPLIT_SHA = '04b5018778321f775a0bf95949a37636090d4df4e3710b16627dfee309408f06'
# Exact previously verified Wav2Vec2 export (not a user-supplied arbitrary NPZ).
EXPORT_SHA = '64cd8d608c5e7e18a0db86193f7b4b26fa08d689aa22e4dab6807b7360bd71c4'
WAVLM_ID = '49725c8a0b4daed276a2ba30dcde37ffab5e220fdf717b1e2be5e0e700ff9d15'
WEIGHTS = (0.75, 0.5)


def require(ok, message):
    if not ok:
        raise ValueError(message)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def read(path):
    return json.loads(Path(path).read_text())


def atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, path)


def save(path, value):
    atomic(path, (json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n').encode())


def npz(path, **arrays):
    b = io.BytesIO()
    np.savez_compressed(b, **arrays)
    atomic(path, b.getvalue())


def probabilities(p, n):
    require(p.shape == (n, 2) and np.isfinite(p).all(), 'Invalid probability shape/finite')
    require(((p >= 0) & (p <= 1)).all() and np.allclose(p.sum(1), 1), 'Invalid probabilities')
    return p


def score(y, p):
    probabilities(p, len(y))
    pred = np.array(['F', 'M'])[p.argmax(1)]
    return dict(n=len(y), correct=int((pred == y).sum()), wrong=int((pred != y).sum()),
                accuracy=float((pred == y).mean()),
                gender_accuracy={g: float((pred[y == g] == g).mean()) for g in ('F', 'M')},
                confusion_F_M=[[int(((y == g) & (pred == h)).sum()) for h in ('F', 'M')] for g in ('F', 'M')])


def fit(x, y, c):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=2000, random_state=42))
    with warnings.catch_warnings(), threadpool_limits(limits=2):
        warnings.simplefilter('error', ConvergenceWarning)
        model.fit(x, y)
    require(list(model.classes_) == ['F', 'M'], 'Class order mismatch')
    return model


def choose(y, p1, p2, folds):
    """A predeclared conservative heuristic, not a statistical significance test."""
    baseline = score(y, p1)
    candidates = []
    for w in WEIGHTS:
        p = w * p1 + (1 - w) * p2
        s = score(y, p)
        deltas = [score(y[folds == k], p[folds == k])['accuracy'] -
                  score(y[folds == k], p1[folds == k])['accuracy'] for k in range(3)]
        eligible = min(deltas) >= 0 and s['accuracy'] - baseline['accuracy'] >= 0.001
        candidates.append(dict(wavlm_weight=w, metrics=s, fold_deltas=deltas, eligible=eligible))
    valid = [r for r in candidates if r['eligible']]
    best = max(valid, key=lambda r: (r['metrics']['correct'], r['wavlm_weight'])) if valid else None
    return dict(wavlm_weight=best['wavlm_weight'] if best else 1.0, baseline=baseline,
                candidates=candidates, rule='3 folds nonnegative gain AND total >=0.1 percentage point; ties prefer WavLM; otherwise baseline',
                scope='Train OOF selection, not an unbiased test; call split is not guaranteed speaker-disjoint')


def metadata(split):
    require(sha(split) == SPLIT_SHA, 'Wrong original split CSV')
    with Path(split).open(newline='') as f:
        rows = sorted(csv.DictReader(f), key=lambda r: r['call_id'])
    require(len(rows) == len({r['call_id'] for r in rows}) == 27985, 'Split coverage')
    require(sum(r['partition'] == 'train' for r in rows) == 22388, 'Train coverage')
    require(sum(r['partition'] == 'internal_validation' for r in rows) == 5597, 'Validation coverage')
    require(all(r['gender'] in ('F', 'M') for r in rows), 'Invalid label')
    return rows


def sources(w1, w2):
    ident = read(w1 / 'identity.json')
    require(digest(ident) == WAVLM_ID, 'Expected the verified L4 WavLM run; select its output directory')
    require(read(w1 / 'completion.json')['status'] == 'complete', 'WavLM run incomplete')
    if w2.is_file():
        require(sha(w2) == EXPORT_SHA, 'Unknown Wav2Vec2 compact export; use original verified export or call_cache directory')
        return ident, dict(kind='export', sha256=EXPORT_SHA)
    summary = read(w2 / 'extraction_summary.json')
    wi = read(w2 / 'call_cache/identity.json')
    require(summary['status'] == 'complete' and summary['identity'] == wi, 'Wav2Vec2 extraction incomplete or identity changed')
    require(summary['completed_calls'] == 27985 and summary['failed_calls'] == 0, 'Wav2Vec2 incomplete coverage')
    require(wi['split_sha256'] == SPLIT_SHA and wi['checkpoint'] == 'facebook/wav2vec2-base', 'Wrong Wav2Vec2 split/model')
    require(wi['revision'] == '0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8', 'Wrong Wav2Vec2 revision')
    return ident, dict(kind='cache', identity=wi, summary_sha256=sha(w2 / 'extraction_summary.json'))


def load_one(row, kind, root, identity):
    cid = row['call_id']
    sub = 'pooled' if kind == 'wavlm' else 'call_cache'
    p = root / sub / cid[:2] / (cid + '.npz')
    require(p.exists(), f'Missing {kind} cache: {p}. Restore the old cache; this lightweight runner does not extract audio.')
    if kind == 'wavlm':
        require(read(p.with_suffix('.hash.json'))['sha256'] == sha(p), 'WavLM cache checksum mismatch')
    with np.load(p, allow_pickle=False) as z:
        require(str(z['call_id'].item()) == cid, 'Cache call ID mismatch')
        if kind == 'wavlm':
            require(str(z['identity'].item()) == digest(identity), 'WavLM cache identity mismatch')
            a = z['features']
            require(a.shape == (4, 2, 1024) and int(z['samples']) > 0, 'WavLM feature shape/coverage')
            a = a[-1, 0].copy()
        else:
            require(str(z['identity_sha256'].item()) == digest(identity), 'Wav2Vec2 cache identity mismatch')
            q = json.loads(str(z['quality'].item()))
            require(q['failed_segments'] == 0 and q['successful_segments'] > 0, 'Wav2Vec2 segment failure')
            a = z['embedding'].copy()
    require(a.shape == ((1024,) if kind == 'wavlm' else (768,)) and a.dtype == np.float32 and np.isfinite(a).all(), 'Invalid feature')
    return a


def matrix(rows, kind, root, identity, out, partition):
    """Drive reads in four threads; committed 512-call chunks allow interruption recovery."""
    blocks = []
    started = time.monotonic()
    for start in range(0, len(rows), 512):
        batch = rows[start:start + 512]
        ids = np.array([r['call_id'] for r in batch])
        p = out / 'packed' / f'{partition}_{kind}_{start:05d}.npz'
        if p.exists():
            require(read(p.with_suffix('.json'))['sha256'] == sha(p), 'Packed cache checksum mismatch')
            with np.load(p, allow_pickle=False) as z:
                require(np.array_equal(z['ids'], ids) and str(z['source'].item()) == digest(identity), 'Packed cache mismatch')
                x = z['x'].copy()
        else:
            with ThreadPoolExecutor(max_workers=4) as pool:
                x = np.stack(list(pool.map(lambda row: load_one(row, kind, root, identity), batch)))
            npz(p, ids=ids, x=x, source=digest(identity))
            save(p.with_suffix('.json'), dict(sha256=sha(p)))
        require(x.shape == (len(batch), 1024 if kind == 'wavlm' else 768) and np.isfinite(x).all(), 'Packed features invalid')
        blocks.append(x)
        done = start + len(batch)
        elapsed = time.monotonic() - started
        print(f'{partition} {kind}: {done}/{len(rows)}; read ETA {elapsed / done * (len(rows)-done) / 60:.1f} min', flush=True)
    return np.concatenate(blocks)


def export_matrix(rows, path):
    with np.load(path, allow_pickle=False) as z:
        ids = z['call_ids'].astype(str)
        require(len(ids) == len(set(ids)) == 27985, 'Export IDs')
        positions = {cid: i for i, cid in enumerate(ids)}
        ix = [positions[r['call_id']] for r in rows]
        require(np.array_equal(z['gender'][ix], [r['gender'] for r in rows]), 'Export labels')
        require(np.array_equal(z['partition'][ix], [r['partition'] for r in rows]), 'Export partitions')
        x = z['embeddings'][ix].copy()
    require(x.shape == (len(rows), 768) and x.dtype == np.float32 and np.isfinite(x).all(), 'Export features')
    return x


def oof(x1, x2, y, ids, out):
    p1, p2 = np.zeros((len(y), 2)), np.zeros((len(y), 2))
    folds = np.full(len(y), -1)
    for k, (tr, dev) in enumerate(StratifiedKFold(3, shuffle=True, random_state=42).split(x1, y)):
        require(not set(ids[tr]) & set(ids[dev]), 'Fold leakage')
        target = out / f'oof_fold_{k}.npz'
        if target.exists():
            require(read(target.with_suffix('.json'))['sha256'] == sha(target), 'OOF checksum mismatch')
            with np.load(target, allow_pickle=False) as z:
                require(np.array_equal(z['ids'], ids[dev]), 'OOF IDs changed')
                a, b = z['wavlm'].copy(), z['wav2vec2'].copy()
        else:
            start = time.monotonic()
            a = fit(x1[tr], y[tr], 0.1).predict_proba(x1[dev])
            b = fit(x2[tr], y[tr], 1.0).predict_proba(x2[dev])
            npz(target, ids=ids[dev], wavlm=a, wav2vec2=b)
            save(target.with_suffix('.json'), dict(sha256=sha(target)))
            print(f'OOF fold {k+1}/3: {(time.monotonic()-start):.1f}s', flush=True)
        p1[dev], p2[dev] = probabilities(a, len(dev)), probabilities(b, len(dev))
        folds[dev] = k
    require((folds >= 0).all(), 'Incomplete OOF')
    return p1, p2, folds


def result_zip(out):
    with zipfile.ZipFile(out / 'ensemble_results.zip.tmp', 'w', zipfile.ZIP_DEFLATED) as z:
        for name in ('identity.json', 'selection.lock.json', 'models.json', 'validation_started.json', 'completion.json', 'REPORT.md', 'validation_predictions.csv'):
            z.write(out / name, name)
    os.replace(out / 'ensemble_results.zip.tmp', out / 'ensemble_results.zip')


def run(args):
    started = time.monotonic()
    w1, w2, out = Path(args.wavlm), Path(args.wav2vec2), Path(args.output)
    rows = metadata(args.split)
    i1, s2 = sources(w1, w2)
    ident = dict(code_sha256=sha(__file__), split_sha256=SPLIT_SHA, wavlm_identity=digest(i1), wav2vec2=s2,
                 versions=dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__),
                 candidates=list(WEIGHTS), folds=3, seed=42, wavlm_C=0.1, wav2vec2_C=1.0)
    out.mkdir(parents=True, exist_ok=True)
    require(out.resolve() != w1.resolve() and out.resolve() != w2.resolve(), 'Use a separate ensemble output')
    if (out / 'identity.json').exists():
        require(read(out / 'identity.json') == ident, 'Resume code/source/environment changed; use a new output directory')
    else:
        require(not any(out.iterdir()), 'Output must be empty for a new run')
        save(out / 'identity.json', ident)
    if (out / 'completion.json').exists():
        result_zip(out)
        print(json.dumps(read(out / 'completion.json'), indent=2)); return
    train = [r for r in rows if r['partition'] == 'train']
    y = np.array([r['gender'] for r in train]); ids = np.array([r['call_id'] for r in train])
    x1 = matrix(train, 'wavlm', w1, i1, out, 'train')
    x2 = export_matrix(train, w2) if s2['kind'] == 'export' else matrix(train, 'wav2vec2', w2, s2['identity'], out, 'train')
    selection_path = out / 'selection.lock.json'
    if not selection_path.exists():
        p1, p2, folds = oof(x1, x2, y, ids, out)
        selection = choose(y, p1, p2, folds)
        selection['identity'] = digest(ident)
        save(selection_path, selection)
    selection = read(selection_path)
    require(selection['identity'] == digest(ident), 'Selection identity mismatch')
    w = selection['wavlm_weight']
    print('Locked WavLM weight:', w, flush=True)
    model_path = out / 'models.joblib'
    if model_path.exists():
        require(read(out / 'models.json')['sha256'] == sha(model_path), 'Final model checksum changed')
        models = joblib.load(model_path)
    else:
        require(not (out / 'validation_started.json').exists(), 'Cannot refit after validation starts')
        models = dict(wavlm=fit(x1, y, 0.1), wav2vec2=fit(x2, y, 1.0) if w < 1 else None,
                      wavlm_weight=w, classes=['F', 'M'], selection_sha256=sha(selection_path))
        b = io.BytesIO(); joblib.dump(models, b); atomic(model_path, b.getvalue())
        save(out / 'models.json', dict(sha256=sha(model_path)))
    require(models['selection_sha256'] == sha(selection_path), 'Selection changed after refit')
    del x1, x2
    marker = dict(selection_sha256=sha(selection_path), model_sha256=sha(model_path))
    if (out / 'validation_started.json').exists():
        require(read(out / 'validation_started.json') == marker, 'Validation identity changed')
    else:
        save(out / 'validation_started.json', marker)
    # Validation features/labels never enter fit, folds, or weight selection.
    val = [r for r in rows if r['partition'] == 'internal_validation']
    yv = np.array([r['gender'] for r in val])
    v1 = matrix(val, 'wavlm', w1, i1, out, 'validation')
    a = probabilities(models['wavlm'].predict_proba(v1), len(val))
    if w < 1:
        v2 = export_matrix(val, w2) if s2['kind'] == 'export' else matrix(val, 'wav2vec2', w2, s2['identity'], out, 'validation')
        b = probabilities(models['wav2vec2'].predict_proba(v2), len(val))
    else:
        b = a
    p = w * a + (1-w) * b
    npz(out / 'validation_predictions.npz', ids=np.array([r['call_id'] for r in val]), gender=yv, probabilities_F_M=p, wavlm_probabilities_F_M=a)
    sio = io.StringIO(); writer = csv.writer(sio, lineterminator='\n')
    writer.writerow(['call_id', 'gender', 'prediction', 'probability_F', 'probability_M'])
    for row, prob in zip(val, p):
        writer.writerow([row['call_id'], row['gender'], 'FM'[int(prob.argmax())], *prob])
    atomic(out / 'validation_predictions.csv', sio.getvalue().encode())
    result = dict(status='complete', identity=digest(ident), wavlm_weight=w,
                  selected=score(yv, p), refit_wavlm_baseline=score(yv, a),
                  session_seconds=time.monotonic()-started,
                  scope='Previously observed Internal Validation; not an independent test. Official Validation unused. No speaker-disjoint guarantee.',
                  historical_wavlm_correct=5490, baseline_reproduced=score(yv, a)['correct'] == 5490)
    atomic(out / 'REPORT.md', (f"# WavLM + Wav2Vec2\n\nWavLM weight: {w}\n\nAccuracy: {result['selected']['accuracy']:.6%}\n\nCorrect: {result['selected']['correct']} / {len(val)}\n\n" + result['scope'] + '\n').encode())
    save(out / 'completion.json', result)
    result_zip(out)
    print(json.dumps(result, indent=2), flush=True)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ('wavlm', 'wav2vec2', 'split', 'output'):
        parser.add_argument('--' + name, required=True)
    args = parser.parse_args()
    # Runtime-local lock releases on Stop/crash; use only one Colab VM per output.
    import fcntl
    lock = open('/tmp/aigo-ensemble-' + hashlib.sha256(str(Path(args.output).resolve()).encode()).hexdigest() + '.lock', 'w')
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    run(args)


## 4. 원본 특징 경로

In [ ]:
# 학습에 사용했던 원본 특징 캐시와 결과 폴더. 새로운 데이터로 대체하지 않는다.
import os
SEARCH_ROOT = Path('/content/drive/MyDrive/DDC-Colab')
WAVLM_OUTPUT = str(SEARCH_ROOT/'wavlm_gender_adaptation_l4_bf16_v2')
WAV2VEC2_SOURCE = str(SEARCH_ROOT/'wav2vec2_frozen_cuda_fp32/wav2vec2_l4_full_embeddings.npz')
ENSEMBLE_OUTPUT = str(SEARCH_ROOT/'wavlm_wav2vec2_ensemble_v1')
assert Path(WAVLM_OUTPUT).is_dir(), 'WavLM original feature cache required'
assert Path(WAV2VEC2_SOURCE).is_file(), 'Wav2Vec2 original feature cache required'
print('WavLM:', WAVLM_OUTPUT, '\nWav2Vec2:', WAV2VEC2_SOURCE, '\n새 결과:', ENSEMBLE_OUTPUT)


WavLM: /content/drive/MyDrive/DDC-Colab/wavlm_gender_adaptation_l4_bf16_v2 
Wav2Vec2: /content/drive/MyDrive/DDC-Colab/wav2vec2_frozen_cuda_fp32/wav2vec2_l4_full_embeddings.npz 
새 결과: /content/drive/MyDrive/DDC-Colab/wavlm_wav2vec2_ensemble_v1


## 5. 학습 및 검증 — 원 실행 출력 보존

In [ ]:
import signal
command = [str(PY), '-u', str(WORK/'run.py'), '--wavlm', WAVLM_OUTPUT,
           '--wav2vec2', WAV2VEC2_SOURCE, '--split', str(WORK/'split_assignments.csv'),
           '--output', ENSEMBLE_OUTPUT]
env = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=True, env=env)
try:
    for line in proc.stdout: print(line, end='', flush=True)
    if proc.wait(): raise RuntimeError('실행 실패. 위 오류를 해결한 뒤 같은 셀로 재개하세요.')
finally:
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try: proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()


train wavlm: 512/22388; read ETA 118.9 min
train wavlm: 1024/22388; read ETA 115.0 min
train wavlm: 1536/22388; read ETA 111.4 min
train wavlm: 2048/22388; read ETA 108.2 min
train wavlm: 2560/22388; read ETA 105.4 min
train wavlm: 3072/22388; read ETA 102.6 min
train wavlm: 3584/22388; read ETA 99.7 min
train wavlm: 4096/22388; read ETA 97.0 min
train wavlm: 4608/22388; read ETA 94.4 min
train wavlm: 5120/22388; read ETA 91.8 min
train wavlm: 5632/22388; read ETA 89.1 min
train wavlm: 6144/22388; read ETA 86.5 min
train wavlm: 6656/22388; read ETA 83.8 min
train wavlm: 7168/22388; read ETA 81.1 min
train wavlm: 7680/22388; read ETA 78.4 min
train wavlm: 8192/22388; read ETA 75.6 min
train wavlm: 8704/22388; read ETA 72.9 min
train wavlm: 9216/22388; read ETA 70.2 min
train wavlm: 9728/22388; read ETA 67.6 min
train wavlm: 10240/22388; read ETA 64.9 min
train wavlm: 10752/22388; read ETA 62.2 min
train wavlm: 11264/22388; read ETA 59.5 min
train wavlm: 11776/22388; read ETA 56.7 min
tr

## 6. 결과 — 원 실행 출력 보존

In [ ]:
# 실제 실행 기록과 최종 결과를 확인한다. 제출과 무관한 브라우저 다운로드 코드는 제외했다.
import json
out = Path(ENSEMBLE_OUTPUT)
result = json.loads((out/'completion.json').read_text())
assert result['status'] == 'complete'
print((out/'REPORT.md').read_text())
print('기존 WavLM 정답 수 재현:', result['baseline_reproduced'])
print('선택 기록:', out/'selection.lock.json')


# WavLM + Wav2Vec2

WavLM weight: 0.75

Accuracy: 98.213329%

Correct: 5497 / 5597

Previously observed Internal Validation; not an independent test. Official Validation unused. No speaker-disjoint guarantee.

기존 WavLM 정답 수 재현: True
선택 기록: /content/drive/MyDrive/DDC-Colab/wavlm_wav2vec2_ensemble_v1/selection.lock.json


## 7. 제출 가중치 변환 코드

학습 완료 모델을 다시 학습하지 않고 두 고정 음성 모델과 Scaler/LR을 `model.pt`로 묶는다. 동봉 체크포인트는 별도 로컬 변환으로 생성했으며 이 셀은 실행 로그로 제시하지 않는다. 이 셀을 실행할 경우 requirements.txt의 패키지가 설치된 환경이 필요하다.

In [ ]:
from pathlib import Path
import hashlib, numpy as np, joblib
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
ORIGINAL_MODELS = Path(ENSEMBLE_OUTPUT) / 'models.joblib'
OUTPUT = Path('/content/mission1_submission')
OUTPUT.mkdir(parents=True, exist_ok=True)
# 원 제출 성능에 대응하는 models.joblib를 사용한다. 확장자 변경이 아닌 파라미터 변환이다.
# 이 셀은 재학습 없이도 helpers/config 셀 다음에 실행할 수 있지만 학습 로그 요건을 대체하지 않는다.
import torch
from transformers import AutoFeatureExtractor, WavLMModel, Wav2Vec2Model
assert sha(ORIGINAL_MODELS) == 'b5a646aa7f3d9d125521969d5eccf922c10799c62bc2655b892a99add24918cc', 'Original model hash mismatch'
original = joblib.load(ORIGINAL_MODELS)  # 직접 학습한 신뢰 가능한 파일만 로드한다.
assert original['classes'] == ['F','M'] and original['wavlm_weight'] == 0.75
checkpoint = dict(format='mission1-ensemble-v1', classes=['F','M'],
                  wavlm_weight=0.75, wavlm_precision='bf16')
for name, repo, revision, cls in [
    ('wavlm','microsoft/wavlm-large','c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c',WavLMModel),
    ('wav2vec2','facebook/wav2vec2-base','0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8',Wav2Vec2Model)]:
    backbone = cls.from_pretrained(repo,revision=revision)
    processor = AutoFeatureExtractor.from_pretrained(repo,revision=revision)
    if name == 'wavlm':
        backbone.config.apply_spec_augment = False
        backbone.config.layerdrop = 0.0
    scaler, lr = original[name].steps[0][1], original[name].steps[1][1]
    assert list(lr.classes_) == ['F','M']
    def tensor(x): return torch.from_numpy(np.asarray(x,dtype=np.float64).copy())
    checkpoint[name] = dict(config=backbone.config.to_dict(), processor=processor.to_dict(),
        weights={k:v.cpu().contiguous() for k,v in backbone.state_dict().items()},
        lr=dict(mean=tensor(scaler.mean_),scale=tensor(scaler.scale_),
                coef=tensor(lr.coef_),intercept=tensor(lr.intercept_)))
    # 확률 계산이 원 sklearn 파이프라인과 일치하는지 합성 특징으로 확인한다.
    from scipy.special import expit
    rng=np.random.default_rng(42)
    x=rng.standard_normal((8,len(scaler.mean_))).astype(np.float32)
    # StandardScaler는 float32 입력의 dtype을 보존한다.
    z=x.copy(); z-=scaler.mean_; z/=scaler.scale_
    male=expit(z@lr.coef_.reshape(-1)+lr.intercept_[0])
    np.testing.assert_allclose(np.column_stack([1-male,male]),original[name].predict_proba(x),rtol=1e-12,atol=1e-12)
    print(name, 'loaded frozen backbone and fitted Scaler/LR')
    del backbone
path=OUTPUT/'model.pt'
torch.save(checkpoint,path)
print('Checkpoint saved:',path,'bytes=',path.stat().st_size)
print('Save this notebook with actual cell outputs; run raw-audio inference verification before submission.')
